In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Optional, Set

import numpy as np
import pandas as pd
from pymongo import MongoClient
import yaml

from mplsoccer import Pitch
import matplotlib.pyplot as plt
from mplsoccer import VerticalPitch

In [2]:
def load_config(path: str | Path = "../config/config.yaml") -> dict[str, Any]:
    path = Path(path)
    if not path.exists():
        path = Path("config/config.yaml")
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


config = load_config()
mongo_config = config["mongo"]

In [3]:
client = MongoClient(mongo_config.get('url'))
db = client[mongo_config.get('db')]

In [4]:
shot_events = pd.DataFrame(list(db[mongo_config.get('collection').get('collection_processed_events')].find({'shot_event': True})))

In [16]:
def shot_statistics(df: pd.DataFrame, stat: str) -> pd.Series:

    counts_value = df[stat].value_counts().reset_index(drop=False)
    counts_normalized = df[stat].value_counts(normalize=True).reset_index(drop=False)
    counts_stats = pd.merge(
        left=counts_value,
        right=counts_normalized,
        left_on=stat, right_on=stat
    )
    return counts_stats 


In [17]:
shot_statistics(shot_events, 'shot_result')

,shot_result,count,proportion
0,Off Target,54775,0.403454
1,Blocked,32854,0.241992
2,On Target,32462,0.239104
3,Goal,15281,0.112555
4,Own Goal,393,0.002895


In [18]:
shot_statistics(shot_events, 'shot_situation')

,shot_situation,count,proportion
0,Open Play,87653,0.645623
1,Set Pieces,30940,0.227894
2,Unknown,8093,0.059610
3,Fastbreak,7195,0.052996
4,Penalty,1491,0.010982
5,Own Goal,393,0.002895


In [22]:
shot_statistics(shot_events, 'shot_zone')

,shot_zone,count,proportion
0,Penalty Area,74408,0.548065
1,Outside of box,51532,0.379568
2,6-yard box,9825,0.072368


In [24]:
shot_statistics(shot_events, 'shot_body_part')

,shot_body_part,count,proportion
0,Right foot,69468,0.511678
1,Left foot,41469,0.305447
2,Head,24344,0.179310
3,Other body parts,484,0.003565


In [29]:
shot_events[shot_events['shot_situation'] == 'Unknown'][['game_id','game_date','season','week','home_team_name','away_team_name','shot_result', 'shot_situation', 'shot_zone', 'shot_body_part']]

,game_id,game_date,season,week,home_team_name,away_team_name,shot_result,shot_situation,shot_zone,shot_body_part
4,327995,2009-08-07 19:30:00,2009-2010,1,Wolfsburg,VfB Stuttgart,Off Target,Unknown,Outside of box,Left foot
16,327995,2009-08-07 19:30:00,2009-2010,1,Wolfsburg,VfB Stuttgart,Off Target,Unknown,Outside of box,Left foot
19,327995,2009-08-07 19:30:00,2009-2010,1,Wolfsburg,VfB Stuttgart,Goal,Unknown,Outside of box,Left foot
20,327995,2009-08-07 19:30:00,2009-2010,1,Wolfsburg,VfB Stuttgart,Blocked,Unknown,6-yard box,Head
29,327997,2009-08-08 14:30:00,2009-2010,1,Borussia Dortmund,FC Koln,On Target,Unknown,6-yard box,Head
...,...,...,...,...,...,...,...,...,...,...
40041,724046,2014-03-29 14:30:00,2013-2014,28,VfB Stuttgart,Borussia Dortmund,Off Target,Unknown,Penalty Area,Head
40044,724046,2014-03-29 14:30:00,2013-2014,28,VfB Stuttgart,Borussia Dortmund,Off Target,Unknown,Penalty Area,Right foot
40045,724046,2014-03-29 14:30:00,2013-2014,28,VfB Stuttgart,Borussia Dortmund,Blocked,Unknown,Penalty Area,Left foot
40051,724046,2014-03-29 14:30:00,2013-2014,28,VfB Stuttgart,Borussia Dortmund,Off Target,Unknown,Outside of box,Right foot


In [30]:
shot_events[shot_events['shot_situation'] == 'Unknown']['game_id'].value_counts()

game_id
420871    17
724033    17
621276    16
621286    15
621392    14
          ..
723811     1
723831     1
723918     1
723943     1
723945     1
Name: count, Length: 1447, dtype: int64

In [32]:
pd.set_option('display.max_columns', None)
shot_events[(shot_events['shot_situation'] == 'Unknown') & (shot_events['game_id'] == 420871)]

,_id,game_id,season,competition_country,competition_name,game_date,game_status,week,home_team_id,home_team_name,away_team_id,away_team_name,event_idx,period,minute,second,expanded_minute,team_id,team,player_id,player,type,outcome_type,x,y,end_x,end_y,goal_mouth_y,goal_mouth_z,blocked_x,blocked_y,related_event_id,related_player_id,stat_event_type,shot_event,pass_event,pass_completed,dribble_event,tackle_attempted_event,interception_event,clearance_event,block_event,offside_event,foul_event,aerial_duel_event,touch_event,loss_possession_event,error_event,save_event,claim_event,punch_event,goalkeeper_event,ball_recovery_event,card_event,substitution_event,shot_goal,shot_on_target,shot_off_target,shot_woodwork,shot_blocked,shot_own_goal,goal_assist,shot_zone_6_yard_box,shot_zone_penalty_area,shot_zone_outside_box,shot_open_play,shot_fastbreak,shot_set_piece,shot_penalty,shot_right_foot,shot_left_foot,shot_head,shot_other_body_part,shot_result,shot_zone,shot_situation,shot_body_part,pass_attempt,pass_incomplete,pass_cross,pass_freekick,pass_corner,pass_through_ball,pass_throw_in,pass_key_pass,pass_key_pass_qualifier,pass_long,pass_short,pass_length,pass_chipped,pass_ground,pass_height,pass_head,pass_feet,pass_body_part,pass_forward,pass_backward,pass_left,pass_right,pass_defensive_third,pass_mid_third,pass_final_third,pass_target_zone,dribble_successful,dribble_unsuccessful,tackle_gained_possession,tackle_did_not_get_possession,tackle_was_dribbled,tackle_result,clearance_head,clearance_feet,clearance_body_part,blocked_shot,blocked_cross,block_type,caught_offside,offside_pass,offside_provoked,offside_type,foul_committed,foul_suffered,foul_type,aerial_duel_won,aerial_duel_lost,aerial_result,dispossessed,turnover,loss_possession_type,error_leading_to_shot,error_leading_to_goal,error_type,keeper_pickup,keeper_sweeper,gk_type,yellow_card,second_yellow_card,red_card,card_type,substitution_on,substitution_off,substitution_type
10701,6a0ca079279178a2ca3eeb22,420871,2010-2011,Germany,Bundesliga,2010-11-13 14:30:00,finished,12,282,FC Koln,134,Borussia M.Gladbach,177,FirstHalf,10,42.0,10,134,Borussia M.Gladbach,41330.0,Marco Reus,MissedShots,Successful,87.2,42.7,None,None,42.4,4.2,NaN,NaN,NaN,NaN,shot,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,Off Target,Penalty Area,Unknown,Right foot,False,False,False,False,False,False,False,False,False,False,False,None,False,False,None,False,False,None,False,False,False,False,False,False,False,None,False,False,False,False,False,None,False,False,None,False,False,None,False,False,False,None,False,False,None,False,False,None,False,False,None,False,False,None,False,False,None,False,False,False,None,False,False,None
10703,6a0ca079279178a2ca3eeb4b,420871,2010-2011,Germany,Bundesliga,2010-11-13 14:30:00,finished,12,282,FC Koln,134,Borussia M.Gladbach,218,FirstHalf,13,56.0,13,134,Borussia M.Gladbach,22825.0,Michael Bradley,MissedShots,Successful,70.8,58.4,None,None,42.0,65.3,NaN,NaN,NaN,NaN,shot,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,Off Target,Outside of box,Unknown,Right foot,False,False,False,False,False,False,False,False,False,False,False,None,False,False,None,False,False,None,False,False,False,False,False,False,False,None,False,False,False,False,False,None,False,False,None,False,False,None,False,False,False,None,False,False,None,False,False,None,False,False,None,False,False,None,False,False,None,False,False,False,None,False,False,None
10707,6a0ca079279178a2ca3eebe5,420871,2010-2011,Germany,Bundesliga,2010-11-13 14:30:00,finished,12,282,FC Koln,134,Borussia M.Gladbach,372,FirstHalf,23,42.0,23,282,FC Koln,19279.0,Milivoje Novakovic,SavedShot,S

In [37]:
for stat in ['shot_result', 'shot_situation', 'shot_zone', 'shot_body_part']:
    display(shot_events[(shot_events['game_id'] == 420871)].groupby('team')[stat].value_counts())



team                 shot_result
Borussia M.Gladbach  Off Target      9
                     Blocked         7
                     Goal            4
FC Koln              Blocked        11
                     Off Target      5
                     On Target       3
Name: count, dtype: int64

team                 shot_situation
Borussia M.Gladbach  Unknown           9
                     Open Play         6
                     Fastbreak         4
                     Set Pieces        1
FC Koln              Unknown           8
                     Open Play         6
                     Set Pieces        5
Name: count, dtype: int64

team                 shot_zone     
Borussia M.Gladbach  Outside of box    13
                     Penalty Area       7
FC Koln              Penalty Area      10
                     Outside of box     8
                     6-yard box         1
Name: count, dtype: int64

team                 shot_body_part
Borussia M.Gladbach  Right foot        18
                     Head               2
FC Koln              Left foot          8
                     Right foot         7
                     Head               4
Name: count, dtype: int64

In [42]:
shot_events[shot_events['shot_situation'] == 'Unknown']['game_id'].value_counts().index.tolist()

[420871,
 724033,
 621276,
 621286,
 621392,
 328126,
 420825,
 421116,
 421179,
 723955,
 328056,
 328169,
 328302,
 420808,
 421234,
 509623,
 509625,
 509711,
 510157,
 621122,
 621306,
 621414,
 723789,
 723817,
 723862,
 723914,
 723919,
 723966,
 723969,
 724037,
 328085,
 328179,
 328219,
 328234,
 328238,
 328266,
 328316,
 420815,
 420999,
 421032,
 421056,
 421230,
 421233,
 421275,
 421294,
 509599,
 509662,
 509758,
 509837,
 510020,
 510113,
 621189,
 621260,
 621278,
 621355,
 723796,
 723816,
 723871,
 723878,
 723971,
 723995,
 724001,
 724002,
 328000,
 328034,
 328055,
 328124,
 328147,
 328225,
 328268,
 328286,
 420785,
 420798,
 420829,
 420849,
 420935,
 421060,
 421101,
 421181,
 421254,
 421276,
 421293,
 421300,
 509546,
 509549,
 509571,
 509726,
 509766,
 509969,
 510024,
 510048,
 510060,
 621167,
 621179,
 621193,
 621202,
 621219,
 621231,
 621290,
 621322,
 621371,
 723745,
 723751,
 723761,
 723828,
 723849,
 723851,
 723860,
 723866,
 723930,
 723947,
 

In [38]:
raw_events = pd.DataFrame(list(db[mongo_config.get('collection').get('collection_raw_events')].find({'game_id': 420871})))

In [39]:
raw_events

,_id,game_id,away_team_id,away_team_name,competition_country,competition_name,game_date,game_status,home_team_id,home_team_name,season,week,period,minute,second,expanded_minute,type,outcome_type,team_id,team,player_id,player,x,y,end_x,end_y,goal_mouth_y,goal_mouth_z,blocked_x,blocked_y,qualifiers,is_touch,is_shot,is_goal,card_type,related_event_id,related_player_id,event_idx
0,6a0aadde0113f547daff6cc4,420871,134,Borussia M.Gladbach,Germany,Bundesliga,2010-11-13 14:30:00,finished,282,FC Koln,2010-2011,12,FirstHalf,0,0.0,0,Start,Successful,282,FC Koln,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,[],False,None,None,NaN,NaN,NaN,0
1,6a0aadde0113f547daff6cc5,420871,134,Borussia M.Gladbach,Germany,Bundesliga,2010-11-13 14:30:00,finished,282,FC Koln,2010-2011,12,FirstHalf,0,0.0,0,Start,Successful,134,Borussia M.Gladbach,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,[],False,None,None,NaN,NaN,NaN,1
2,6a0aadde0113f547daff6cc6,420871,134,Borussia M.Gladbach,Germany,Bundesliga,2010-11-13 14:30:00,finished,282,FC Koln,2010-2011,12,FirstHalf,0,0.0,0,Pass,Successful,282,FC Koln,19279.0,Milivoje Novakovic,49.6,50.6,49.0,53.4,NaN,NaN,NaN,NaN,"[{'type': {'displayName': 'Length', 'value': 2...",True,None,None,NaN,NaN,NaN,2
3,6a0aadde0113f547daff6cc7,420871,134,Borussia M.Gladbach,Germany,Bundesliga,2010-11-13 14:30:00,finished,282,FC Koln,2010-2011,12,FirstHalf,0,2.0,0,Pass,Successful,282,FC Koln,6040.0,Lukas Podolski,49.0,53.4,37.8,46.3,NaN,NaN,NaN,NaN,"[{'type': {'displayName': 'Angle', 'value': 21...",True,None,None,NaN,NaN,NaN,3
4,6a0aadde0113f547daff6cc8,420871,134,Borussia M.Gladbach,Germany,Bundesliga,2010-11-13 14:30:00,finished,282,FC Koln,2010-2011,12,FirstHalf,0,5.0,0,Pass,Successful,134,Borussia M.Gladbach,21564.0,Tobias Levels,19.4,28.0,26.0,11.4,NaN,NaN,NaN,NaN,"[{'type': {'displayName': 'PassEndY', 'value':...",True,None,None,NaN,NaN,NaN,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1331,6a0aadde0113f547daff71f7,420871,134,Borussia M.Gladbach,Germany,Bundesliga,2010-11-13 14:30:00,finished,282,FC Koln,2010-2011,12,SecondHalf,90,36.0,92,End,Successful,134,Borussia M.Gladbach,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,[],False,None,None,NaN,NaN,NaN,1331
1332,6a0aadde0113f547daff71f8,420871,134,Borussia M.Gladbach,Germany,Bundesliga,2010-11-13 14:30:00,finished,282,FC Koln,2010-2011,12,PostGame,0,0.0,2,End,Successful,282,FC Koln,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,[],False,None,None,NaN,NaN,NaN,1332
1333,6a0aadde0113f547daff71f9,420871,134,Borussia M.Gladbach,Germany,Bundesliga,2010-11-13 14:30:00,finished,282,FC Koln,2010-2011,12,PostGame,0,0.0,2,End,Successful,134,Borussia M.Gladbach,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,[],False,None,None,NaN,NaN,NaN,1333
1334,6a0aadde0113f547daff71fa,420871,134,Borussia M.Gladbach,Germany,Bundesliga,2010-11-13 14:30:00,finished,282,FC Koln,2010-2011,12,PreMatch,0,0.0,0,FormationSet,Successful,134,Borussia M.Gladbach,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,[{'type': {'displayName': 'TeamPlayerFormation...,False,None,None,NaN,NaN,NaN,1334


In [ ]:
shot_events.groupby("shot_situation")["shot_goal"].agg(["count", "sum", "mean"])


In [ ]:
shot_events['shot_zone'].value_counts(normalize=True, dropna=False)

In [ ]:
shot_events.groupby("shot_zone")["shot_goal"].agg(["count", "sum", "mean"])


In [ ]:
shot_events['shot_body_part'].value_counts(normalize=True, dropna=False)

In [ ]:
shot_events.groupby("shot_body_part")["shot_goal"].agg(["count", "sum", "mean"])


In [ ]:
def season_goals(df: pd.DataFrame, season: str,
                 category: str, plot_type: str = "Goals") -> pd.DataFrame:
    
    season_df = df[df['season'] == season]
    if plot_type == "Goals":
        df = season_df[season_df["shot_goal"]].reset_index(drop=True)
        pitch = VerticalPitch(
                    half=True,
                    pitch_type="opta",
                    pitch_color='#aabb97',
                    line_color='white', 
                    stripe=True)
    
    else:
        df = season_df[season_df["shot_goal"] == False]
        pitch = Pitch(
                    pitch_type="opta",
                    pitch_color='#aabb97',
                    line_color='white', 
                    stripe=True)

    
    if category == "shot_body_part":
        colors = {
            "Right foot": "#2563EB",        # blue
            "Left foot": "#16A34A",         # green
            "Head": "#F97316",              # orange
            "Other body parts": "#7C3AED",  # purple
        }

    elif category == "shot_situation":
        colors = {
            "Open Play": "#2563EB",     # blue
            "Set Pieces": "#DC2626",    # red
            "Fastbreak": "#F59E0B",     # amber
            "Penalty": "#7C3AED",       # purple
            "Own Goal": "#111827",      # near black
            "Unknown": "#6B7280",       # gray
        }

    elif category == "shot_zone":
        colors = {
            "6-yard box": "#DC2626",       # red
            "Penalty Area": "#2563EB",     # blue
            "Outside of box": "#16A34A",   # green
            "Unknown": "#6B7280",          # gray
        }

    


    fig, ax = pitch.draw(figsize=(10, 7))

    for value, group in df.groupby(category, dropna=False):
        label = "Unknown" if pd.isna(value) else value
        pitch.scatter(
            group["x"],
            group["y"],
            ax=ax,
            s=45,
            color=colors.get(value, "#7f7f7f"),
            alpha=0.6,
            edgecolors="none",
            label=label,
        )

    ax.set_title(f"Goal Locations by {category} | {season}", fontsize=16)
    ax.legend(loc="upper left", frameon=True)

    plt.show()


In [ ]:
seasons = shot_events['season'].unique()

for season in seasons:
    season_goals(df=shot_events, season=season, category="shot_body_part")
    season_goals(df=shot_events, season=season, category="shot_situation")
    season_goals(df=shot_events, season=season, category="shot_zone")


In [ ]:

for season in seasons:
    season_goals(df=shot_events, season=season, category="shot_body_part", plot_type="Non-Goals")
    season_goals(df=shot_events, season=season, category="shot_situation", plot_type="Non-Goals")
    season_goals(df=shot_events, season=season, category="shot_zone", plot_type="Non-Goals")
